# So sánh toàn bộ độ đo — 6 cổ phiếu × 2 giai đoạn × 3 chiến lược

Notebook độc lập này chỉ đọc CSV kết quả từ `application/EIDT_Project/final_result`. Mọi xử lý, kiểm tra và trực quan hóa đều nằm trong notebook; không gọi file Python ngoài.

- Deep SARSA UCB-VAE và Epsilon-Greedy: mean ± sample standard deviation trên 20 seed.
- Buy-and-Hold: một baseline xác định (`n=1`, SD bằng 0).
- Độ đo: Final Profit, ROI, ARR, Volatility, Sharpe Ratio, Maximum Drawdown và Constraint Violations.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICKERS = ("ACB", "FPT", "GAS", "HPG", "SSI", "VCB")
PERIODS = ("GOOD", "BAD")
MODEL_LABELS = {
    "UNCERTAINTY_AWARE_UCB_009": "UCB-VAE",
    "EPSILON_GREEDY": "Epsilon-Greedy",
    "BUY_AND_HOLD": "Buy-and-Hold",
}
COLORS = {"UCB-VAE": "#2369BD", "Epsilon-Greedy": "#E6862D", "Buy-and-Hold": "#4E9F6D"}
METRICS = {
    "profit": ("Final Profit", "test_profit"),
    "roi": ("ROI (%)", "test_roi"),
    "arr": ("ARR (%)", "test_arr"),
    "volatility": ("Volatility (%)", "test_volatility"),
    "sharpe": ("Sharpe Ratio", "test_sharpe"),
    "max_drawdown": ("Maximum Drawdown (%)", "test_max_drawdown"),
    "violations": ("Constraint Violations", "test_violations"),
}

def find_project_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "application" / "EIDT_Project" / "final_result").exists(): return candidate
    raise FileNotFoundError("Không tìm thấy project root")

ROOT = find_project_root()
SOURCE = ROOT / "application" / "EIDT_Project" / "final_result"
OUTPUT = ROOT / "application" / "FAIR_2026" / "metric_comparison"
OUTPUT.mkdir(parents=True, exist_ok=True)

def period_dir(ticker, period):
    found = [p for p in (SOURCE / ticker).iterdir() if p.is_dir() and p.name.upper() == period]
    if len(found) != 1: raise FileNotFoundError(f"Không xác định được folder {ticker}/{period}")
    return found[0]


In [ ]:
rows = []
for ticker in TICKERS:
    for period in PERIODS:
        folder = period_dir(ticker, period)
        rl = pd.read_csv(folder / "rl_metrics.csv")
        for model_key in ("UNCERTAINTY_AWARE_UCB_009", "EPSILON_GREEDY"):
            group = rl.loc[rl.model_key == model_key].copy()
            if len(group) != 20 or group.seed.nunique() != 20:
                raise ValueError(f"{ticker}/{period}/{model_key} không có đúng 20 seed")
            for metric, (_, column) in METRICS.items():
                values = pd.to_numeric(group[column], errors="raise")
                if not np.isfinite(values).all(): raise ValueError(f"Giá trị không hữu hạn: {ticker}/{period}/{column}")
                rows.append({"ticker": ticker, "period": period, "model_key": model_key,
                    "strategy": MODEL_LABELS[model_key], "metric": metric, "n": 20,
                    "mean": float(values.mean()), "std": float(values.std(ddof=1))})
        hold = pd.read_csv(folder / "buy_and_hold_metrics.csv")
        for metric, (_, column) in METRICS.items():
            value = float(pd.to_numeric(hold[column], errors="raise").iloc[0])
            rows.append({"ticker": ticker, "period": period, "model_key": "BUY_AND_HOLD",
                "strategy": MODEL_LABELS["BUY_AND_HOLD"], "metric": metric, "n": 1,
                "mean": value, "std": 0.0})

summary = pd.DataFrame(rows)
expected = len(TICKERS) * len(PERIODS) * len(MODEL_LABELS) * len(METRICS)
assert len(summary) == expected
summary.to_csv(OUTPUT / "all_metrics_6stocks_good_bad.csv", index=False)
print(f"Đã kiểm tra {len(summary)} ô thống kê ({expected} kỳ vọng).")
summary.head(10)


In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.titleweight": "bold"})
strategies = list(MODEL_LABELS.values())
x = np.arange(len(PERIODS), dtype=float)
width = 0.23
saved_figures = []

for metric, (title, _) in METRICS.items():
    frame = summary[summary.metric == metric]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.7))
    fig.subplots_adjust(top=.84, bottom=.09, hspace=.38, wspace=.18)
    for ax, ticker in zip(axes.ravel(), TICKERS):
        ticker_frame = frame[frame.ticker == ticker]
        for strategy_index, strategy in enumerate(strategies):
            ordered = ticker_frame[ticker_frame.strategy == strategy].set_index("period").loc[list(PERIODS)]
            offset = (strategy_index - 1) * width
            bars = ax.bar(x + offset, ordered["mean"], width, yerr=ordered["std"],
                          capsize=3.5, color=COLORS[strategy], edgecolor="white", linewidth=.7,
                          label=strategy, zorder=3, error_kw={"elinewidth": 1, "capthick": 1})
            for bar, mean in zip(bars, ordered["mean"]):
                ax.annotate(f"{mean:.2f}", (bar.get_x() + bar.get_width()/2, bar.get_height()),
                            xytext=(0, 3 if mean >= 0 else -4), textcoords="offset points",
                            ha="center", va="bottom" if mean >= 0 else "top", fontsize=7)
        ax.axhline(0, color="#444", linewidth=.8)
        ax.set_xticks(x, PERIODS); ax.set_title(ticker); ax.set_ylabel(title)
        ax.grid(axis="y", alpha=.25, zorder=0); ax.spines[["top", "right"]].set_visible(False)
    handles, legend_labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, ncol=3, loc="upper center", bbox_to_anchor=(.5, .925), frameon=False)
    fig.suptitle(f"{title} Across Six Stocks and Two Market Periods", fontsize=16, fontweight="bold", y=.985)
    fig.text(.5, .018, "Each stock panel uses its own y-scale. Bars: mean; whiskers: sample SD over 20 RL seeds. Buy-and-Hold: n=1.",
             ha="center", fontsize=9, color="#444")
    png = OUTPUT / f"{metric}_comparison_6stocks_good_bad.png"
    pdf = OUTPUT / f"{metric}_comparison_6stocks_good_bad.pdf"
    fig.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    plt.show(); plt.close(fig)
    saved_figures.extend([png, pdf]); print("Saved:", png)

print(f"Đã tạo {len(METRICS)} hình PNG và {len(METRICS)} PDF độc lập.")


## Kết quả đã tạo

- Final Profit: ![Profit](metric_comparison/profit_comparison_6stocks_good_bad.png)
- ROI: ![ROI](metric_comparison/roi_comparison_6stocks_good_bad.png)
- ARR: ![ARR](metric_comparison/arr_comparison_6stocks_good_bad.png)
- Volatility: ![Volatility](metric_comparison/volatility_comparison_6stocks_good_bad.png)
- Sharpe Ratio: ![Sharpe](metric_comparison/sharpe_comparison_6stocks_good_bad.png)
- Maximum Drawdown: ![MDD](metric_comparison/max_drawdown_comparison_6stocks_good_bad.png)
- Constraint Violations: ![Violations](metric_comparison/violations_comparison_6stocks_good_bad.png)
